# Ingest data

The Ingest API reads files from storage, converts them, and adds them to your Organization's STAC catalog. You describe an ingestion in a version 2 ingestion spec.

An ingestion has two phases:

1. Create the ingestion. The platform scans the input locations. When the scan ends, the status is `READY`.
2. Start the ingestion. The platform processes the files. At the end, the status is `DONE` or `FAILED`.

You must be an admin of your Organization to create and start an ingestion. The input files must already be in your Organization bucket or in an External bucket. The SDK 1.0 cannot upload files.

The Ingest API is at `<prescient_endpoint_url>/ingest/`. The `IngestClient` uses the access token of a `PrescientClient`.

In [ ]:
from prescient_sdk import (
    IngestClient,
    IngestResource,
    IngestSpec,
    LiveStatus,
    PrescientClient,
)
from prescient_sdk.ingest_models import ErrorSeverity

client = IngestClient(PrescientClient(env_file="config.env"))

# True if the Ingest API answers.
client.check()

## Write a spec

A version 2 spec has `version: "2.0.0"` and no `user` field. The Organization comes from your API key.

A location `path` with no scheme, for example `example-rasters`, is a path in your Organization bucket. The platform reads input files from `uploads/example-rasters/` and writes outputs under `assets/`. A path that starts with `s3://` is an External bucket.

`IngestSpec.problems()` lists the problems that the SDK can find before you submit. An empty list means that the SDK found no problems. The server checks the rest.

In [ ]:
spec = IngestSpec.from_bytes(
    b"""
version: "2.0.0"
name: Example rasters

locations:
  rasters:
    path: example-rasters

tasks:
  to_cog:
    type: transform
    input_location: rasters
    input_format: GTiff
    output_format: COG
    output_path_template: "example-rasters/{STAC_ITEM_ID}/{STAC_ITEM_ID}.tif"

stac:
  collection:
    id: example-rasters
    title: Example rasters
    description: GeoTIFF files converted to Cloud Optimized GeoTIFF.
    license: CC-BY-4.0
  item:
    assets:
      data:
        asset_src: to_cog
        type: image/tiff; application=geotiff; profile=cloud-optimized
        title: Data
        roles: [data]
"""
)

spec.problems()

To read a spec from a YAML file, use `IngestSpec.from_file("spec.yaml")`.

## Create the ingestion and wait for the scan

`IngestResource.create` submits the spec. `LiveStatus` shows the status while you wait. `wait_until_ready()` returns when the scan ends.

If the server rejects the spec, `create` raises `IngestSpecError`. If another Organization owns the STAC collection, it raises `CollectionOwnershipError`. The `detail` attribute of the error holds the server's reason.

In [ ]:
ing = IngestResource.create(client, spec)

with LiveStatus(ing):
    ing.wait_until_ready()

## Check what the scan found

Read the errors even if the scan succeeded. Warnings are in the same list.

In [ ]:
for err in ing.errors():
    print(f"[{err.severity.value}] {err.description}")

print(f"{len(ing.input_files())} input files")
print(f"{len(ing.output_files())} planned outputs")

## Start the ingestion

Start the ingestion only if the status is `READY` and there are no errors other than warnings. `wait_until_done()` returns when the status is `DONE` or `FAILED`.

In [ ]:
from prescient_sdk.ingest_models import Status

problems = [e for e in ing.errors() if e.severity != ErrorSeverity.WARNING]

if ing.status is Status.READY and not problems:
    with LiveStatus(ing):
        ing.start().wait_until_done()

ing.status

## Check the output files

Each output file has its own status, so you can see which files failed.

In [ ]:
for out in ing.output_files():
    print(f"{out.task_name}: {out.status.value} ({out.stac_item_id})")

## Open an existing ingestion

`IngestResource.from_id` gets an ingestion by its ID. `IngestResource.list_all` lists your Organization's ingestions, newest first. In Jupyter, a resource in the last line of a cell shows a summary.

In [ ]:
ing = IngestResource.from_id(client, id=42)
ing

## Ingest new data with a batch

A batch scans the same locations again and picks up new files. It uses the spec of the ingestion. `create_batch()` returns a `BatchResource`, which has the same wait and start methods.

In [ ]:
batch = ing.create_batch()

with LiveStatus(batch):
    batch.wait_until_ready()
    problems = [e for e in batch.errors() if e.severity != ErrorSeverity.WARNING]
    if batch.status is Status.READY and not problems:
        batch.start().wait_until_done()

batch.status

## Call the API directly

`IngestClient` has one method for each Ingest API endpoint, for example `create_ingestion`, `get_ingestion`, `start_ingestion`, `create_batch` and `get_batch_errors`. The methods return pydantic models. They do not wait for a status; poll with `get_ingestion` or `get_batch`.

An error response raises `IngestAPIError`. It is a subclass of `requests.HTTPError`.

In [ ]:
ingestion = client.create_ingestion(spec.to_bytes())
ingestion = client.get_ingestion(ingestion.id)
ingestion.id, ingestion.status